In [2]:
import torch
import numpy as np
import pandas as pd
from torchinfo import summary

In [33]:
class Dummy(torch.nn.Module):
    def __init__(self):
        super(Dummy, self).__init__()
        
        # Initialize linear layers (shared)
        self.linear_layers_shared = torch.nn.ModuleList()
        self.linear_layers_shared.add_module('Linear_Dropout_0_shared', torch.nn.Dropout(0.2))
        self.linear_layers_shared.add_module('Linear_Linear_0_shared', torch.nn.Linear(in_features=1, out_features=3))
        self.linear_layers_shared.add_module('Linear_LReLU_0_shared', torch.nn.LeakyReLU())
            
        # Initialize linear layers (endpoint specific)
        linear_layers_endpoint_dict_i = dict()
        linear_layers_endpoint_dict_i['Xer'] = torch.nn.ModuleList()
        linear_layers_endpoint_dict_i['Dys'] = torch.nn.ModuleList()
        self.linear_layers_endpoint_dict = torch.nn.ModuleDict(linear_layers_endpoint_dict_i)

        for tox in ['Xer', 'Dys']:
            linear_layers_endpoint = self.linear_layers_endpoint_dict[tox]
            linear_layers_endpoint.add_module('{}_Linear_Dropout_0'.format(tox), torch.nn.Dropout(0.1))
            linear_layers_endpoint.add_module('{}_Linear_Linear_0'.format(tox), torch.nn.Linear(in_features=3, out_features=2))
            linear_layers_endpoint.add_module('{}_Linear_LReLU_0'.format(tox), torch.nn.LeakyReLU())
            linear_layers_endpoint.add_module('{}_Output'.format(tox), torch.nn.Linear(in_features=2, out_features=1))

    def forward(self, x):
        # Linear layers (shared)
        for layer in self.linear_layers_shared:
            x = layer(x)
        
        # Clone tensor (preserving the gradient)
        x_dict = dict()
        for tox in ['Xer', 'Dys']:
            x_dict[tox] = x.clone()
            
        # Linear layers (endpoint specific)
        for tox in ['Xer', 'Dys']:
            for layer in self.linear_layers_endpoint_dict[tox]:
                x_dict[tox] = layer(x_dict[tox])
        
        return x_dict

In [40]:
batch_size = 4
model = Dummy()
summary(model, input_size=(batch_size, 1))

Layer (type:depth-idx)                   Output Shape              Param #
Dummy                                    --                        --
├─ModuleList: 1-1                        --                        --
├─ModuleDict: 1                          --                        --
│    └─ModuleList: 2-1                   --                        --
│    └─ModuleList: 2-2                   --                        --
├─ModuleList: 1-1                        --                        --
│    └─Dropout: 2-3                      [4, 1]                    --
│    └─Linear: 2-4                       [4, 3]                    6
│    └─LeakyReLU: 2-5                    [4, 3]                    --
├─ModuleDict: 1                          --                        --
│    └─ModuleList: 2-1                   --                        --
│    │    └─Dropout: 3-1                 [4, 3]                    --
│    │    └─Linear: 3-2                  [4, 2]                    8
│    │    └─Leaky

In [41]:
device = 'cpu'
model.to(device)
inp = torch.rand((batch_size, 1), device=device)

oup = model(inp)

In [48]:
named_layers = dict(model.named_parameters())
print('named_layers.keys(): {}\n'.format(named_layers.keys()))
print('named_layers: {}'.format(named_layers))

named_layers.keys(): dict_keys(['linear_layers_shared.Linear_Linear_0_shared.weight', 'linear_layers_shared.Linear_Linear_0_shared.bias', 'linear_layers_endpoint_dict.Xer.Xer_Linear_Linear_0.weight', 'linear_layers_endpoint_dict.Xer.Xer_Linear_Linear_0.bias', 'linear_layers_endpoint_dict.Xer.Xer_Output.weight', 'linear_layers_endpoint_dict.Xer.Xer_Output.bias', 'linear_layers_endpoint_dict.Dys.Dys_Linear_Linear_0.weight', 'linear_layers_endpoint_dict.Dys.Dys_Linear_Linear_0.bias', 'linear_layers_endpoint_dict.Dys.Dys_Output.weight', 'linear_layers_endpoint_dict.Dys.Dys_Output.bias'])

named_layers: {'linear_layers_shared.Linear_Linear_0_shared.weight': Parameter containing:
tensor([[-0.0682],
        [ 0.8382],
        [ 0.6734]]), 'linear_layers_shared.Linear_Linear_0_shared.bias': Parameter containing:
tensor([-0.7075,  0.3607, -0.6243]), 'linear_layers_endpoint_dict.Xer.Xer_Linear_Linear_0.weight': Parameter containing:
tensor([[ 0.0447,  0.3131, -0.1375],
        [-0.5035, -0.0585,

In [42]:
for name, para in model.named_parameters():
    print("-"*20)
    print(f"name: {name}")
    print("values: ")
    print(para)

--------------------
name: linear_layers_shared.Linear_Linear_0_shared.weight
values: 
Parameter containing:
tensor([[-0.0682],
        [ 0.8382],
        [ 0.6734]], requires_grad=True)
--------------------
name: linear_layers_shared.Linear_Linear_0_shared.bias
values: 
Parameter containing:
tensor([-0.7075,  0.3607, -0.6243], requires_grad=True)
--------------------
name: linear_layers_endpoint_dict.Xer.Xer_Linear_Linear_0.weight
values: 
Parameter containing:
tensor([[ 0.0447,  0.3131, -0.1375],
        [-0.5035, -0.0585, -0.4896]], requires_grad=True)
--------------------
name: linear_layers_endpoint_dict.Xer.Xer_Linear_Linear_0.bias
values: 
Parameter containing:
tensor([ 0.3441, -0.1331], requires_grad=True)
--------------------
name: linear_layers_endpoint_dict.Xer.Xer_Output.weight
values: 
Parameter containing:
tensor([[0.5107, 0.1591]], requires_grad=True)
--------------------
name: linear_layers_endpoint_dict.Xer.Xer_Output.bias
values: 
Parameter containing:
tensor([0.2210]

In [49]:
# Freeze shared layers
for name, param in model.named_parameters():
    if param.requires_grad and '_shared_' in name:
        param.requires_grad = False
        
endpoint = 'Xerostomia_M12'
# Freeze toxicity layer
for name, param in model.named_parameters():
    if param.requires_grad and '{}'.format(endpoint) in name:
        param.requires_grad = False

In [50]:
for name, para in model.named_parameters():
    print("-"*20)
    print(f"name: {name}")
    print(para)

--------------------
name: linear_layers_shared.Linear_Linear_0_shared.weight
Parameter containing:
tensor([[-0.0682],
        [ 0.8382],
        [ 0.6734]])
--------------------
name: linear_layers_shared.Linear_Linear_0_shared.bias
Parameter containing:
tensor([-0.7075,  0.3607, -0.6243])
--------------------
name: linear_layers_endpoint_dict.Xer.Xer_Linear_Linear_0.weight
Parameter containing:
tensor([[ 0.0447,  0.3131, -0.1375],
        [-0.5035, -0.0585, -0.4896]], requires_grad=True)
--------------------
name: linear_layers_endpoint_dict.Xer.Xer_Linear_Linear_0.bias
Parameter containing:
tensor([ 0.3441, -0.1331], requires_grad=True)
--------------------
name: linear_layers_endpoint_dict.Xer.Xer_Output.weight
Parameter containing:
tensor([[0.5107, 0.1591]], requires_grad=True)
--------------------
name: linear_layers_endpoint_dict.Xer.Xer_Output.bias
Parameter containing:
tensor([0.2210], requires_grad=True)
--------------------
name: linear_layers_endpoint_dict.Dys.Dys_Linear_Li